# Chicago Taxi Market Analysis
**TripleTen Data Science Program · Sprint 7 — SQL, EDA & Hypothesis Testing**

**Objective:** Analyze Chicago's taxi market using SQL-sourced data and test whether bad weather significantly increases ride duration on the Loop-to-O'Hare route.

**Two phases:**
1. EDA — market dominance across 64 taxi companies and dropoff patterns across 94 neighborhoods  
2. Hypothesis Testing — does bad weather increase Saturday Loop-to-O'Hare ride duration?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

## 1. Load Data

Three datasets extracted from SQL queries against a Chicago taxi database (`trips`, `cabs`, `weather_records`).

In [ ]:
companies    = pd.read_csv('project_sql_result_01.csv')
neighborhoods = pd.read_csv('project_sql_result_04.csv')
rides        = pd.read_csv('project_sql_result_07.csv')

print('Companies dataset:    ', companies.shape)
print('Neighborhoods dataset:', neighborhoods.shape)
print('Rides dataset:        ', rides.shape)

In [ ]:
companies.head()

In [ ]:
neighborhoods.head()

In [ ]:
rides.head()

## 2. Data Quality Check

In [ ]:
for name, df in [('Companies', companies), ('Neighborhoods', neighborhoods), ('Rides', rides)]:
    print(f'--- {name} ---')
    print(df.dtypes)
    print('Missing values:')
    print(df.isnull().sum())
    print()

In [ ]:
rides['start_ts'] = pd.to_datetime(rides['start_ts'])
rides['duration_minutes'] = rides['duration_seconds'] / 60
print('Rides date range:', rides['start_ts'].min(), 'to', rides['start_ts'].max())
print('Weather conditions:', rides['weather_conditions'].value_counts().to_dict())

## 3. EDA — Taxi Company Market Share

In [ ]:
top10_companies = companies.nlargest(10, 'trips_amount')
top10_companies[['company_name', 'trips_amount']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top10_companies['company_name'][::-1],
    top10_companies['trips_amount'][::-1],
    color=sns.color_palette('Blues_d', 10)
)

for bar in bars:
    ax.text(
        bar.get_width() + 100,
        bar.get_y() + bar.get_height() / 2,
        f'{int(bar.get_width()):,}',
        va='center', fontsize=9
    )

ax.set_xlabel('Number of Trips (Nov 15–16, 2017)')
ax.set_title('Top 10 Chicago Taxi Companies by Ride Volume', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
total_trips   = companies['trips_amount'].sum()
flash_trips   = companies.loc[companies['company_name'] == 'Flash Cab', 'trips_amount'].values[0]
flash_share   = flash_trips / total_trips * 100

top10_total   = top10_companies['trips_amount'].sum()
top10_share   = top10_total / total_trips * 100

print(f'Total trips (all companies):  {total_trips:,}')
print(f'Flash Cab trips:              {flash_trips:,}  ({flash_share:.1f}% market share)')
print(f'Top-10 combined share:        {top10_share:.1f}%')

**Observation:** Flash Cab is the dominant player, leading second-place Taxi Affiliation Services by ~71%. The top-10 companies together represent highly concentrated market share, suggesting significant barriers to entry or brand loyalty.

## 4. EDA — Neighborhood Dropoff Patterns

In [ ]:
top10_hoods = neighborhoods.nlargest(10, 'average_trips')
top10_hoods[['dropoff_location_name', 'average_trips']]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top10_hoods['dropoff_location_name'][::-1],
    top10_hoods['average_trips'][::-1],
    color=sns.color_palette('Greens_d', 10)
)

for bar in bars:
    ax.text(
        bar.get_width() + 50,
        bar.get_y() + bar.get_height() / 2,
        f'{bar.get_width():,.1f}',
        va='center', fontsize=9
    )

ax.set_xlabel('Average Daily Dropoffs (November 2017)')
ax.set_title('Top 10 Chicago Neighborhoods by Average Taxi Dropoffs', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

**Observation:** The Loop dominates all other neighborhoods with 10,727 average daily dropoffs — nearly 13% more than second-place River North. The top three (Loop, River North, Streeterville) are all in Chicago's dense downtown core, which together account for the majority of taxi activity.

## 5. Hypothesis Testing — Weather Effect on Loop-to-O'Hare Ride Duration

**Research Question:** Does bad weather (rain/storm) significantly increase ride duration on the Loop-to-O'Hare route on Saturdays?

**Hypotheses:**
- H₀: Average ride duration is the same in good and bad weather
- H₁: Average ride duration is longer in bad weather
- Significance level: α = 0.05

**Test:** Two-sample one-tailed t-test (independent samples, equal variance not assumed)

In [ ]:
good_weather = rides.loc[rides['weather_conditions'] == 'Good', 'duration_seconds']
bad_weather  = rides.loc[rides['weather_conditions'] == 'Bad',  'duration_seconds']

print('Good weather rides:', len(good_weather))
print('Bad weather rides: ', len(bad_weather))
print()
print('Good weather — mean: {:.1f}s ({:.1f} min), std: {:.1f}s'.format(
    good_weather.mean(), good_weather.mean()/60, good_weather.std()))
print('Bad weather  — mean: {:.1f}s ({:.1f} min), std: {:.1f}s'.format(
    bad_weather.mean(), bad_weather.mean()/60, bad_weather.std()))
print()
diff = bad_weather.mean() - good_weather.mean()
print(f'Difference: +{diff:.1f}s (+{diff/60:.1f} min, +{diff/good_weather.mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution comparison
axes[0].hist(good_weather / 60, bins=40, alpha=0.6, label='Good weather', color='steelblue', edgecolor='white')
axes[0].hist(bad_weather  / 60, bins=40, alpha=0.6, label='Bad weather',  color='tomato',    edgecolor='white')
axes[0].axvline(good_weather.mean() / 60, color='steelblue', linestyle='--', linewidth=1.5, label=f'Good mean: {good_weather.mean()/60:.1f} min')
axes[0].axvline(bad_weather.mean()  / 60, color='tomato',    linestyle='--', linewidth=1.5, label=f'Bad mean:  {bad_weather.mean()/60:.1f} min')
axes[0].set_xlabel('Ride Duration (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ride Duration by Weather')
axes[0].legend()

# Box plot
ride_plot = rides[['weather_conditions', 'duration_minutes']].copy()
ride_plot['weather_conditions'] = pd.Categorical(ride_plot['weather_conditions'], categories=['Good', 'Bad'], ordered=True)
ride_plot.boxplot(column='duration_minutes', by='weather_conditions', ax=axes[1],
                  boxprops=dict(color='steelblue'), medianprops=dict(color='tomato', linewidth=2))
axes[1].set_xlabel('Weather Conditions')
axes[1].set_ylabel('Ride Duration (minutes)')
axes[1].set_title('Ride Duration by Weather (Loop → O\'Hare, Saturdays)')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
t_stat, p_two_tailed = stats.ttest_ind(bad_weather, good_weather, equal_var=False)
p_one_tailed = p_two_tailed / 2  # one-tailed: H₁ is directional (bad > good)

print('=== Hypothesis Test Results ===')
print(f't-statistic:     {t_stat:.4f}')
print(f'p-value (2-tail): {p_two_tailed:.6f}')
print(f'p-value (1-tail): {p_one_tailed:.6f}')
print(f'alpha:            0.05')
print()
if p_one_tailed < 0.05:
    print('Decision: REJECT H₀')
    print('Conclusion: Bad weather significantly increases Loop-to-O\'Hare ride duration.')
else:
    print('Decision: FAIL TO REJECT H₀')

## 6. Final Summary

### EDA Findings
- **Flash Cab** is the dominant taxi company, leading 63 competitors by ~71% in ride volume
- The **Loop** neighborhood leads all 94 dropoff zones — reflecting Chicago's commercial and transit hub
- Market concentration is high: top-3 companies and top-3 neighborhoods account for a disproportionate share of total trips

### Hypothesis Test Conclusion

| Metric | Value |
|--------|-------|
| t-statistic | 6.8405 |
| p-value | ≈ 0.000000 |
| Decision | **Reject H₀** |
| Practical Effect | +6.9 min (+20.6%) |

Bad weather **statistically significantly** increases Loop-to-O'Hare Saturday ride duration (p ≪ α = 0.05). Rides in bad weather average **40.5 minutes** vs **33.6 minutes** in good weather — a 6.9-minute (+20.6%) increase that is both statistically significant and practically meaningful for airport travelers.